# 📊 Scientific Evaluation Metrics for DRL Navigation

To match the "Challenging Conventions" paper, we must transition from "It works" to "It is efficient and safe."

## 1. The Sway Index (Behavioral Quality)
The paper defines the Sway Index to measure the smoothness of navigation. 
**Calculation:** It is the integral of the absolute change in angular velocity ($|\omega_t - \omega_{t-1}|$). 
**Goal:** A lower Sway Index indicates a more "human-like" and stable path.

## 2. Distance and Time (Efficiency)
By integrating the distance between $(x_t, y_t)$ and $(x_{t+1}, y_{t+1})$ from Odometry, we calculate the exact path length. This allows us to compare our DRL agent against the "Optimal Path" (Straight line).

## 3. Collision Categorization
To implement the "Static vs Dynamic" metric, the environment must identify if the `min_obstacle_distance` came from a wall or a moving actor in Gazebo. This is critical for assessing the robot's "Reactive Intelligence."

| Metric | Currently in your Code? | Status |
|--------|------------------------|--------|
| Success/Reward Recording | Yes (dqn_agent.py & training_analysis.py) | ✅ |
| Success Rate (%) | Approximated in training_analysis.py | ⚠️ |
| Static vs. Dynamic Collision | No. Your code treats all obstacles the same. | ❌ |
| Timeout Rate | Logged as text, but not calculated as a % metric. | ❌ |
| Average Distance/Time | No. You track "Steps," but not physical meters. | ❌ |
| Sway Index | No. This requires tracking angular velocity changes. | ❌ |

In [ ]:
#inside dqn_environment.py

# Inside __init__
self.total_distance = 0.0
self.start_time = 0.0
self.last_action_angular = 0.0
self.total_sway = 0.0

# Inside reset_environment
self.total_distance = 0.0
self.start_time = self.get_clock().now()
self.total_sway = 0.0

# Inside calculate_state / step logic
# Calculate change in angular velocity for Sway Index
sway = abs(current_angular_vel - self.last_action_angular)
self.total_sway += sway

: 

In [ ]:
#inside dqn_test.py
results = {
    'success_rate': 0,
    'avg_dist': [],
    'avg_time': [],
    'sway_index': []
}
# After 100 episodes, calculate the averages.

## 5. Optimized Hyperparameters (Stability & Memory)
**Goal:** Prevent "Catastrophic Forgetting" and ensure the agent can handle both Static and Dynamic obstacles in Stage 3.

### 🧠 Why these values? (Scientific Evidence)
According to **Figure 4 and Table 1** of the paper:
1. **Batch Size (1024):** Small batches (128) cause the reward curve to oscillate (jitter). A large batch provides a stable gradient, leading to a 97-99% success rate.
2. **Replay Buffer (1,000,000):** If the buffer is too small (1e+5), the robot "forgets" how to avoid walls as soon as it starts learning how to reach the goal. A 1e+6 buffer preserves a "long-term memory" of all scenarios.
3. **Learning Rate (3e-4):** This is the "Sweet Spot." Anything higher (1e-3) causes the brain to diverge; anything lower (1e-4) trains too slowly to finish within 40 hours.

### 🛠️ Required Code Changes in `dqn_agent.py`
Update the `__init__` method of the `DQNAgent` class with these precise values:

```python
# --- Paper-Aligned Hyperparameters ---
self.batch_size = 1024        # Optimized for RTX 3050 Ti stability
self.buffer_size = 1000000     # 1e+6: Prevents "Catastrophic Forgetting"
self.learning_rate = 0.0003    # 3e-4: The optimal rate for Stage 3
self.discount_factor = 0.99    # Gamma: Focus on long-term rewards

# --- Apply to Objects ---
self.memory = collections.deque(maxlen=self.buffer_size)
self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)

## 7. Laser Scan Density (Spatial Perception)
**Goal:** Increase the number of LiDAR samples to exactly 40 points to ensure the robot can "see" wall corners and small obstacles.

### 🧠 Why 40 Samples? (Scientific Evidence)
According to **Figure 6** in the paper:
1. **Blind Spots (< 40):** With 10 or 20 samples, the gaps between laser beams are too wide. A table leg or a wall corner can "hide" between the beams, leading to 60-70% collision rates.
2. **Diminishing Returns (> 40):** Using 360 or 720 samples makes the Neural Network input too large. This slows down training without improving the success rate. 
3. **The Result:** 40 samples achieved the **94% success rate** and the best average time/distance.

### 🛠️ Required Code Changes in `dqn_environment.py`

You must modify the `get_scan` or `scan_callback` logic to downsample the raw 360-degree LiDAR data into 40 evenly spaced points.

**Update your Laser Processing logic:**
```python
# Inside dqn_environment.py (scan_callback or equivalent)
def process_laser_scan(self, msg):
    # msg.ranges typically contains 360 values (1 per degree)
    raw_data = np.array(msg.ranges)
    
    # Replace Infinity/NaN with max range (3.5m)
    raw_data = np.where(np.isinf(raw_data), 3.5, raw_data)
    raw_data = np.where(np.isnan(raw_data), 3.5, raw_data)
    
    # DOWNSAMPLE to 40 points (360 / 40 = every 9th degree)
    # This matches the paper's 'evenly distributed' requirement
    indices = np.linspace(0, len(raw_data) - 1, 40).astype(int)
    self.front_ranges = raw_data[indices].tolist()

## 8. Laser Scan Density (Spatial Resolution)
**Goal:** Increase the number of LiDAR samples to exactly 40 points to ensure the robot can "see" wall corners and small obstacles.

### 🧠 Why 40 Samples? (Scientific Evidence)
According to **Figure 6** and **Table 2** in the paper:
1. **The "Blind" Robot (< 40):** With only 10 or 20 samples, the gaps between laser beams are too wide. A table leg or a sharp wall corner can "hide" in the gap, leading to 60-70% collision rates in Stage 3.
2. **Efficiency (40):** 40 samples achieved a **94% success rate**. It provides the best balance between seeing obstacles and keeping the neural network input small for fast training.
3. **Diminishing Returns (> 40):** Using 360 samples doesn't help more; it just makes the "Brain" (Neural Network) larger and slower to train.

### 🛠️ Required Code Changes in `dqn_environment.py`
We must modify the laser processing to take 40 evenly spaced points from the raw 360-degree scan.

**Update the scan processing logic:**
```python
# In dqn_environment.py
def get_scan(self):
    # Assume scan_msg is the raw 360-degree data from Gazebo
    scan_data = np.array(self.scan_msg.ranges)
    
    # Replace Inf/NaN with max range (3.5m)
    scan_data = np.where(np.isinf(scan_data), 3.5, scan_data)
    scan_data = np.where(np.isnan(scan_data), 3.5, scan_data)
    
    # DOWNSAMPLE: Pick 40 points evenly across the 360 degrees
    # (360 degrees / 40 samples = one sample every 9 degrees)
    indices = np.linspace(0, len(scan_data) - 1, 40).astype(int)
    self.front_ranges = scan_data[indices].tolist()

# 🤖 Project Context: TurtleBot3 DQN Optimization
**Target Stage:** Stage 3 (Obstacle Maze)
**Scientific Reference:** "Challenging Conventions" (2024) - Autonomous Navigation using DRL.

## 📌 Project Overview
We are moving our TurtleBot3 DQN implementation from a standard/basic reward structure to the **Model RG (Reward Gradient)** architecture defined in the 2024 research paper. Our goal is to achieve a 95%+ success rate in Stage 3.

## 🛠️ Current Implementation Status
- **Agent:** PyTorch-based DQN with 3 Hidden Layers (512, 256, 128).
- **State Space:** 182-dimensional (Distance, Angle, and 180 Laser Scan rays).
- **Action Space:** 5 Discrete Angular Velocities [1.5, 0.75, 0.0, -0.75, -1.5].
- **Environment:** ROS 2 Humble/Foxy Gazebo simulation.

## 🎯 The "Model RG" Roadmap
To collaborate with me on this project, please follow these "Model RG" specifications from the paper:

### 1. Terminal Reward System
- **Goal Success:** +2500.0 (Old: 100)
- **Collision/Timeout:** -2000.0 (Old: -50)
- *Reason:* Large terminal rewards provide a stronger gradient for the neural network.

### 2. The Composite Reward Formula (RG)
The reward at each step must be: `Reward = rd1 + r_alpha + r_va1 + r_ob1`
- **rd1 (Distance):** `(prev_dist - current_dist) * 100` -> Rewards progress toward goal.
- **r_alpha (Heading):** `math.cos(goal_angle)` -> Smoothly rewards facing the goal.
- **r_va1 (Steering):** `-0.1` penalty if `action != straight` -> Reduces "Sway Index."
- **r_ob1 (Obstacles):** `-1.0` penalty if `min_scan < 0.25m` -> Creates a proactive "Danger Zone."

## 💬 How to help us:
1. **Code Audits:** When I share `dqn_environment.py` or `dqn_agent.py`, check if our logic strictly follows the RG/RH models from the paper.
2. **Hyperparameter Tuning:** Suggest changes to `epsilon_decay` or `learning_rate` if our Success Rate (from the CSV log) isn't climbing.
3. **Bug Fixing:** Help debug ROS 2 service calls between the Agent and the Environment.
4. **Data Analysis:** If I provide our `training_log.csv`, help me interpret why the Loss is spiking or why the Q-values are saturating.

---
**Current Priority:** Implementing the `prev_goal_distance` tracking in `dqn_environment.py` to enable the `rd1` distance reward.